In [ ]:
# Setup for Kaggle or Google Colab environments
# Run this cell to install any missing dependencies!
!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn mplsoccer tqdm

In [ ]:
# -----------------------------------------------------------
# Kaggle / Colab Environment Setup & Core Functions
# -----------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (20, 6) # wider for 3 subplots

# Configuration Constants
class config:
    ID_COLS = [
        "event_id", "match_id", "season_id",
        "team_id", "team_name", "player_id", "player_name",
        "period", "minute", "second", "play_pattern",
    ]
    RANDOM_STATE = 42
    K_RANGE = range(2, 11)
    KMEANS_N_INIT = 10
    DISTANCE_METRIC = "euclidean"

# -----------------------------------------------------------
# Clustering Functions
# -----------------------------------------------------------
def run_kmeans(X, k, init="k-means++"):
    model = KMeans(
        n_clusters=k,
        init=init,
        random_state=config.RANDOM_STATE,
        n_init=config.KMEANS_N_INIT,
    )
    labels = model.fit_predict(X)
    inertia = model.inertia_
    return labels, inertia, model

def run_kmeans_sweep(X, k_range=config.K_RANGE):
    results = {}
    for k in k_range:
        results[k] = run_kmeans(X, k)
    return results

def describe_centroids(X_unscaled, labels):
    df = X_unscaled.copy()
    df["Cluster"] = labels
    return df.groupby("Cluster").mean()

# -----------------------------------------------------------
# Selection Metrics Functions
# -----------------------------------------------------------
def elbow_curve(X, k_range=config.K_RANGE):
    results = run_kmeans_sweep(X, k_range)
    data = [{"k": k, "inertia": res[1]} for k, res in results.items()]
    return pd.DataFrame(data)

def suggest_k_elbow(curve):
    n_points = len(curve)
    if n_points < 3:
        return curve["k"].iloc[0]
        
    all_coords = curve[["k", "inertia"]].values
    first_point = all_coords[0]
    last_point = all_coords[-1]
    
    line_vec = last_point - first_point
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    
    vec_from_first = all_coords - first_point
    scalar_proj = np.sum(vec_from_first * line_vec_norm, axis=1)
    
    vec_proj = np.outer(scalar_proj, line_vec_norm)
    vec_to_line = vec_from_first - vec_proj
    dist_to_line = np.linalg.norm(vec_to_line, axis=1)
    
    best_idx = np.argmax(dist_to_line)
    return int(curve["k"].iloc[best_idx])

def silhouette_by_k(X, k_range=config.K_RANGE):
    results = run_kmeans_sweep(X, k_range)
    data = []
    for k, (labels, inertia, model) in results.items():
        if k > 1:
            score = silhouette_score(X, labels, metric=config.DISTANCE_METRIC)
        else:
            score = -1.0
        data.append({"k": k, "silhouette": score})
    return pd.DataFrame(data)

def gap_statistic(X, k_range=config.K_RANGE, n_refs=5):
    # Convert to numpy for calculations
    X_arr = X.values if isinstance(X, pd.DataFrame) else X
    
    # Bounding box of original data
    mins = np.min(X_arr, axis=0)
    maxs = np.max(X_arr, axis=0)
    
    gap_values = []
    for k in k_range:
        _, original_inertia, _ = run_kmeans(X_arr, k)
        
        ref_inertias = []
        for _ in range(n_refs):
            # Generate uniform random data in bounding box
            random_data = np.random.uniform(mins, maxs, size=X_arr.shape)
            _, ref_inertia, _ = run_kmeans(random_data, k)
            ref_inertias.append(ref_inertia)
            
        mean_ref_log_inertia = np.mean(np.log(ref_inertias))
        gap = mean_ref_log_inertia - np.log(original_inertia)
        gap_values.append({"k": k, "gap_value": gap})
        
    return pd.DataFrame(gap_values)

def summarize_k_selection(elbow, silhouette, gap=None):
    df = elbow.merge(silhouette, on="k")
    if gap is not None:
        df = df.merge(gap, on="k")
        
    best_elbow = suggest_k_elbow(elbow)
    best_silhouette = silhouette["k"].iloc[np.argmax(silhouette["silhouette"].values)]
    
    suggestions = {
        "k": [best_elbow, best_silhouette],
        "method": ["Elbow (Max dist)", "Silhouette (Max score)"]
    }
    
    if gap is not None and "gap_value" in gap.columns:
        best_gap = gap["k"].iloc[np.argmax(gap["gap_value"].values)]
        suggestions["k"].append(best_gap)
        suggestions["method"].append("Gap (Max value)")
        
    print("\nSuggested K by method:")
    for m, k in zip(suggestions["method"], suggestions["k"]):
        print(f" - {m}: k={k}")
        
    return df

In [ ]:
# -----------------------------------------------------------
# Preprocessing Configuration & Logic
# -----------------------------------------------------------

# Configurations
config.HIDDEN_COLS = ["outcome", "statsbomb_xg", "end_location_x", "end_location_y"]
config.NUMERIC_FEATURES = [
    "location_x", "location_y", "distance_to_goal", "angle_to_goal",
    "n_teammates_in_frame", "n_opponents_in_frame", "keeper_x", "keeper_y"
]
config.CATEGORICAL_FEATURES = ["body_part", "technique", "shot_type"]
config.BOOL_FLAGS = ["under_pressure", "first_time", "open_goal", "aerial_won"]

# Load raw dataset (Output from Task 1)
try:
    df = pd.read_csv("../../data/extract-feature/extract_feature_worldcup_2018_2022_raw.csv")
except FileNotFoundError:
    print("Local data folder not found. Assuming Kaggle/Colab current directory.")
    df = pd.read_csv("shots_worldcup_2018_2022_raw.csv")  # Kaggle working directory typically

print("Raw shape:", df.shape)

# 1. Split columns
ids = df[config.ID_COLS].copy()
y_hidden = df[config.HIDDEN_COLS].copy()
X = df[config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES + config.BOOL_FLAGS].copy()

# 2. Impute missing values
if "keeper_x" in X.columns:
    X["keeper_x"] = X["keeper_x"].fillna(120.0)
if "keeper_y" in X.columns:
    X["keeper_y"] = X["keeper_y"].fillna(40.0)
for col in config.BOOL_FLAGS:
    if col in X.columns:
        X[col] = X[col].fillna(0)

# 3. One-hot encoding
X_encoded = pd.get_dummies(X, columns=config.CATEGORICAL_FEATURES, drop_first=False)
for col in config.BOOL_FLAGS:
    if col in X_encoded.columns:
        X_encoded[col] = X_encoded[col].astype(int)
for col in X_encoded.columns:
    if X_encoded[col].dtype == bool:
        X_encoded[col] = X_encoded[col].astype(int)

# 4. Create unscaled, standard-scaled, and robust-scaled versions
X_unscaled = X_encoded.copy()

std_scaler = StandardScaler()
X_std_arr = std_scaler.fit_transform(X_encoded)
X_standard = pd.DataFrame(X_std_arr, columns=X_encoded.columns, index=X_encoded.index)

robust_scaler = RobustScaler()
X_robust_arr = robust_scaler.fit_transform(X_encoded)
X_robust = pd.DataFrame(X_robust_arr, columns=X_encoded.columns, index=X_encoded.index)

# 5. Prepare variables for modeling
X_train_robust = X_robust.copy()
X_train_standard = X_standard.copy()
X_train_unscaled = X_unscaled.copy()

print("Robust Scaled shape:", X_train_robust.shape)
print("Standard Scaled shape:", X_train_standard.shape)
print("Unscaled shape:", X_train_unscaled.shape)

In [ ]:
# 1. Elbow Method Comparison
df_elbow_robust = elbow_curve(X_train_robust)
best_k_elbow_robust = suggest_k_elbow(df_elbow_robust)

df_elbow_standard = elbow_curve(X_train_standard)
best_k_elbow_standard = suggest_k_elbow(df_elbow_standard)

df_elbow_unscaled = elbow_curve(X_train_unscaled)
best_k_elbow_unscaled = suggest_k_elbow(df_elbow_unscaled)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(df_elbow_robust["k"], df_elbow_robust["inertia"], marker="o", linestyle="-", color="blue")
axes[0].axvline(best_k_elbow_robust, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_robust}")
axes[0].set_title("Elbow Curve (ROBUST)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].legend()

axes[1].plot(df_elbow_standard["k"], df_elbow_standard["inertia"], marker="o", linestyle="-", color="green")
axes[1].axvline(best_k_elbow_standard, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_standard}")
axes[1].set_title("Elbow Curve (STANDARD)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Inertia")
axes[1].legend()

axes[2].plot(df_elbow_unscaled["k"], df_elbow_unscaled["inertia"], marker="o", linestyle="-", color="orange")
axes[2].axvline(best_k_elbow_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_unscaled}")
axes[2].set_title("Elbow Curve (UNSCALED)")
axes[2].set_xlabel("Number of Clusters (k)")
axes[2].set_ylabel("Inertia")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2. Silhouette Score Comparison
df_sil_robust = silhouette_by_k(X_train_robust)
best_k_sil_robust = df_sil_robust["k"].iloc[np.argmax(df_sil_robust["silhouette"].values)]

df_sil_standard = silhouette_by_k(X_train_standard)
best_k_sil_standard = df_sil_standard["k"].iloc[np.argmax(df_sil_standard["silhouette"].values)]

df_sil_unscaled = silhouette_by_k(X_train_unscaled)
best_k_sil_unscaled = df_sil_unscaled["k"].iloc[np.argmax(df_sil_unscaled["silhouette"].values)]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(df_sil_robust["k"], df_sil_robust["silhouette"], marker="s", color="blue", linestyle="-")
axes[0].axvline(best_k_sil_robust, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_robust}")
axes[0].set_title("Silhouette Curve (ROBUST)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Avg Silhouette Score")
axes[0].legend()

axes[1].plot(df_sil_standard["k"], df_sil_standard["silhouette"], marker="s", color="green", linestyle="-")
axes[1].axvline(best_k_sil_standard, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_standard}")
axes[1].set_title("Silhouette Curve (STANDARD)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Avg Silhouette Score")
axes[1].legend()

axes[2].plot(df_sil_unscaled["k"], df_sil_unscaled["silhouette"], marker="s", color="orange", linestyle="-")
axes[2].axvline(best_k_sil_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_unscaled}")
axes[2].set_title("Silhouette Curve (UNSCALED)")
axes[2].set_xlabel("Number of Clusters (k)")
axes[2].set_ylabel("Avg Silhouette Score")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 3. Gap Statistic Comparison
print("Calculating Gap Statistics (this might take a few moments)...")
df_gap_robust = gap_statistic(X_train_robust)
best_k_gap_robust = df_gap_robust["k"].iloc[np.argmax(df_gap_robust["gap_value"].values)]

df_gap_standard = gap_statistic(X_train_standard)
best_k_gap_standard = df_gap_standard["k"].iloc[np.argmax(df_gap_standard["gap_value"].values)]

df_gap_unscaled = gap_statistic(X_train_unscaled)
best_k_gap_unscaled = df_gap_unscaled["k"].iloc[np.argmax(df_gap_unscaled["gap_value"].values)]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(df_gap_robust["k"], df_gap_robust["gap_value"], marker="^", color="blue", linestyle="-")
axes[0].axvline(best_k_gap_robust, color="r", linestyle="--", label=f"Suggested K = {best_k_gap_robust}")
axes[0].set_title("Gap Statistic (ROBUST)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Gap Value")
axes[0].legend()

axes[1].plot(df_gap_standard["k"], df_gap_standard["gap_value"], marker="^", color="green", linestyle="-")
axes[1].axvline(best_k_gap_standard, color="r", linestyle="--", label=f"Suggested K = {best_k_gap_standard}")
axes[1].set_title("Gap Statistic (STANDARD)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Gap Value")
axes[1].legend()

axes[2].plot(df_gap_unscaled["k"], df_gap_unscaled["gap_value"], marker="^", color="orange", linestyle="-")
axes[2].axvline(best_k_gap_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_gap_unscaled}")
axes[2].set_title("Gap Statistic (UNSCALED)")
axes[2].set_xlabel("Number of Clusters (k)")
axes[2].set_ylabel("Gap Value")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary Tables
print("=== ROBUST SCALED SUMMARY ===")
summary_robust = summarize_k_selection(df_elbow_robust, df_sil_robust, df_gap_robust)
display(summary_robust)

print("\n=== STANDARD SCALED SUMMARY ===")
summary_standard = summarize_k_selection(df_elbow_standard, df_sil_standard, df_gap_standard)
display(summary_standard)

print("\n=== UNSCALED SUMMARY ===")
summary_unscaled = summarize_k_selection(df_elbow_unscaled, df_sil_unscaled, df_gap_unscaled)
display(summary_unscaled)

In [ ]:
# 4. K-Means Centroid Initialization Comparison (k-means++ vs random/forgy)
# We test with K=5 to see differences in performance and inertia
test_k = 5
print(f"Comparing Centroid Initialization for K = {test_k}\n")

def compare_initialization(X, name):
    _, inertia_kmeanspp, model_kmeanspp = run_kmeans(X, k=test_k, init="k-means++")
    _, inertia_random, model_random = run_kmeans(X, k=test_k, init="random")
    
    print(f"--- {name} Data ---")
    print(f"  k-means++ : Inertia = {inertia_kmeanspp:.2f}, Iterations = {model_kmeanspp.n_iter_}")
    print(f"  random    : Inertia = {inertia_random:.2f}, Iterations = {model_random.n_iter_}\n")
    
compare_initialization(X_train_robust, "ROBUST SCALED")
compare_initialization(X_train_standard, "STANDARD SCALED")
compare_initialization(X_train_unscaled, "UNSCALED")

In [ ]:
# Based on the curves, choose the best K for the branches
def pick_best_k(df_gap, df_sil, df_elbow):
    k1 = df_gap["k"].iloc[np.argmax(df_gap["gap_value"].values)]
    k2 = df_sil["k"].iloc[np.argmax(df_sil["silhouette"].values)]
    k3 = suggest_k_elbow(df_elbow)
    # Most common or just take the max if no consensus
    return int(np.max([k1, k2, k3]))

CHOSEN_K_ROBUST = pick_best_k(df_gap_robust, df_sil_robust, df_elbow_robust)
CHOSEN_K_STANDARD = pick_best_k(df_gap_standard, df_sil_standard, df_elbow_standard)
CHOSEN_K_UNSCALED = pick_best_k(df_gap_unscaled, df_sil_unscaled, df_elbow_unscaled)

print(f"Applying ROBUST SCALED K-Means with K = {CHOSEN_K_ROBUST}")
print(f"Applying STANDARD SCALED K-Means with K = {CHOSEN_K_STANDARD}")
print(f"Applying UNSCALED K-Means with K = {CHOSEN_K_UNSCALED}")

labels_robust, _, model_robust = run_kmeans(X_train_robust, k=CHOSEN_K_ROBUST)
labels_standard, _, model_standard = run_kmeans(X_train_standard, k=CHOSEN_K_STANDARD)
labels_unscaled, _, model_unscaled = run_kmeans(X_train_unscaled, k=CHOSEN_K_UNSCALED)

# Save labels
pd.DataFrame({"cluster_id": labels_robust}).to_csv("labels_robust.csv", index=False)
pd.DataFrame({"cluster_id": labels_standard}).to_csv("labels_standard.csv", index=False)
pd.DataFrame({"cluster_id": labels_unscaled}).to_csv("labels_unscaled.csv", index=False)
print("Saved all labels arrays to local directory.")

In [ ]:
# Describe Centroids physically
# Note: X_unscaled has no ID columns, it represents the features used.
X_unscaled_features = X_unscaled.copy()

print("=== ROBUST SCALED CENTROIDS (in original units) ===")
centroids_physical_robust = describe_centroids(X_unscaled_features, labels_robust)
display(centroids_physical_robust)

print("\n=== STANDARD SCALED CENTROIDS (in original units) ===")
centroids_physical_standard = describe_centroids(X_unscaled_features, labels_standard)
display(centroids_physical_standard)

print("\n=== UNSCALED CENTROIDS ===")
centroids_physical_unscaled = describe_centroids(X_unscaled_features, labels_unscaled)
display(centroids_physical_unscaled)

In [ ]:
# Plotting the clusters geographically side-by-side
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

def draw_pitch(ax, title, labels):
    ax.plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
    ax.plot([120, 120], [36, 44], color="red", linewidth=4)
    sns.scatterplot(
        ax=ax, x=X_unscaled["location_x"], y=X_unscaled["location_y"], 
        hue=labels, palette="tab10", alpha=0.6, s=20
    )
    ax.set_title(title)
    ax.legend(title="Cluster")

draw_pitch(axes[0], f"ROBUST (K={CHOSEN_K_ROBUST})", labels_robust)
draw_pitch(axes[1], f"STANDARD (K={CHOSEN_K_STANDARD})", labels_standard)
draw_pitch(axes[2], f"UNSCALED (K={CHOSEN_K_UNSCALED})", labels_unscaled)

plt.tight_layout()
plt.show()